## Search for URLs

In [ ]:
import requests

def search(query: str):
    url = "http://localhost:8888/search"
    params = {
        "q": query,
        "format": "json"
    }

    r = requests.get(url, params=params)
    data = r.json()

    urls = [r["url"] for r in data["results"][:15]]
    return urls

In [80]:
urls = search("How LangChain can be used for creating AI Agents?")
urls

['https://www.designveloper.com/blog/how-to-build-ai-agents-with-langchain/',
 'https://academy.langchain.com/courses/foundation-introduction-to-langchain-python',
 'https://www.langchain.com/langchain',
 'https://www.freecodecamp.org/news/build-ai-agent-with-langchain-fastapi-and-sevalla/',
 'https://medium.com/@dhruvmali999/how-to-use-langchain-to-build-smart-ai-agents-with-real-examples-d15d77853109',
 'https://www.reddit.com/r/Rag/comments/1pjliq0/i_made_a_complete_tutorial_on_building_ai_agents/',
 'https://www.geeksforgeeks.org/artificial-intelligence/building-ai-agents-using-langchain-and-langgraph/',
 'https://medium.com/@dvasquez.422/building-a-simple-ai-agent-1e2f2b369b25',
 'https://agentsapis.com/langchain/build-ai-agent-with-langchain/',
 'https://www.youtube.com/watch?v=J7j5tCB_y4w',
 'https://www.guvi.in/blog/build-ai-agents-with-langchain-v1/',
 'https://www.codecademy.com/article/agentic-ai-with-langchain-langgraph',
 'https://www.getzep.com/ai-agents/langchain-agents-

## Download webpages

In [12]:
def fetch_pages(url):
    headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
    }
    r = requests.get(url, timeout=10, headers=headers)
    return r.text

In [16]:
html = fetch_pages(urls[0])
html

'<!doctype html><html lang="en"><head><title data-rh="true">How Large Language Models Work. From zero to ChatGPT | by Andreas Stöffelbauer | Medium | Data Science + AI at Microsoft</title><meta data-rh="true" charset="utf-8"/><meta data-rh="true" name="viewport" content="width=device-width,minimum-scale=1,initial-scale=1,maximum-scale=1"/><meta data-rh="true" name="theme-color" content="#000000"/><meta data-rh="true" name="twitter:app:name:iphone" content="Medium"/><meta data-rh="true" name="twitter:app:id:iphone" content="828256236"/><meta data-rh="true" property="al:ios:app_name" content="Medium"/><meta data-rh="true" property="al:ios:app_store_id" content="828256236"/><meta data-rh="true" property="al:android:package" content="com.medium.reader"/><meta data-rh="true" property="fb:app_id" content="542599432471018"/><meta data-rh="true" property="og:site_name" content="Medium"/><meta data-rh="true" name="apple-itunes-app" content="app-id=828256236, app-argument=/data-science-at-micros

## Extract clean content

In [17]:
import trafilatura

def extract_text(html):
    return trafilatura.extract(html)

In [19]:
text = extract_text(html)
text

'How Large Language Models work\nFrom zero to ChatGPT\nThanks to Large Language Models (or LLMs for short), Artificial Intelligence has now caught the attention of pretty much everyone. ChatGPT, possibly the most famous LLM, has immediately skyrocketed in popularity due to the fact that natural language is such a, well, natural interface that has made the recent breakthroughs in Artificial Intelligence accessible to everyone. Nevertheless, how LLMs work is still less commonly understood, unless you are a Data Scientist or in another AI-related role. In this article, I will try to change that.\nAdmittedly, that’s an ambitious goal. After all, the powerful LLMs we have today are a culmination of decades of research in AI. Unfortunately, most articles covering them are one of two kinds: They are either very technical and assume a lot of prior knowledge, or they are so trivial that you don’t end up knowing more than before.\nThis article is meant to strike a balance between these two appro

## Split text into chunks

In [26]:
def chunk_text(text, size=500):
    words = text.split()
    for i in range(0, len(words), size):
        yield " ".join(words[i:i+size])

In [27]:
chunks = list(chunk_text(text))
chunks

['How Large Language Models work From zero to ChatGPT Thanks to Large Language Models (or LLMs for short), Artificial Intelligence has now caught the attention of pretty much everyone. ChatGPT, possibly the most famous LLM, has immediately skyrocketed in popularity due to the fact that natural language is such a, well, natural interface that has made the recent breakthroughs in Artificial Intelligence accessible to everyone. Nevertheless, how LLMs work is still less commonly understood, unless you are a Data Scientist or in another AI-related role. In this article, I will try to change that. Admittedly, that’s an ambitious goal. After all, the powerful LLMs we have today are a culmination of decades of research in AI. Unfortunately, most articles covering them are one of two kinds: They are either very technical and assume a lot of prior knowledge, or they are so trivial that you don’t end up knowing more than before. This article is meant to strike a balance between these two approach

## Create embeddings

In [60]:
from sentence_transformers import SentenceTransformer, CrossEncoder

def create_embeddings(chunks):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    return model.encode(chunks)

## Store vectors

In [36]:
import faiss
import numpy as np

def build_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

## Retrieve relevant chunks

In [ ]:
def retrieve(index, chunks, query, top_k=10):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    query_vec = model.encode([query])
    D, I = index.search(query_vec, top_k)

    return [chunks[i] for i in I[0]]

## Re-ranking with cross-encoders

In [56]:
def rerank(chunks, query, top_k=5):
    cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    pairs = [[query, passage] for passage in chunks]
    scores = cross_encoder.predict(pairs)
    reranked = [c for _, c in sorted(zip(scores, chunks), reverse=True)]

    return reranked[:top_k]

In [33]:
relevant_chunks

['we inquired about are already in the LLM’s working memory, which makes it easier to answer correctly. Conclusion Before I wrap things up, I want to answer a question I asked earlier in the article. Is the LLM really just predicting the next word or is there more to it? Some researchers are arguing for the latter, saying that to become so good at next-word-prediction in any context, the LLM must actually have acquired a compressed understanding of the world internally. Not, as others argue, that the model has simply learned to memorize and copy patterns seen during training, with no actual understanding of language, the world, or anything else. There is probably no clear right or wrong between those two sides at this point; it may just be a different way of looking at the same thing. Clearly these LLMs are proving to be very useful and show impressive knowledge and reasoning capabilities, and maybe even show some sparks of general intelligence. But whether or to what extent that resem

## Passing to LLM

In [53]:
from ollama import Client

def generate_with_llm(context, query):
    prompt = f"""
    You are an expert AI assistant. Use this context below only to answer the question clearly and concisely. Also add the link of the sources and mention inline number of the source.
    Context:
    {context}
    
    Question:
    {query}
    """
    client = Client(host="http://110.226.127.67:11434")

    stream = client.chat(
        model="mistral-small:22b",

        messages=[{
            "role": "user",
            "content": prompt
        }],
        stream=True
    )

    answer = ""
    for chunk in stream:
        print(chunk.message.content, end="", flush=True)
        answer += chunk.message.content
    
    return answer

## Final Pipeline

In [66]:
def process_url(url):
    """Process a single URL: fetch -> extract -> chunk"""
    html = fetch_pages(url)
    text = extract_text(html)
    if text:
        chunks = list(chunk_text(text))
        return url, chunks
    return url, []

In [75]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_pipline(query):
    urls = search(query)

    all_chunks = []
    sources = []

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_url, url): url for url in urls}

        for future in as_completed(futures):
            url, chunks = future.result()
            if chunks:
                all_chunks.extend(chunks)
                sources.extend([url]*len(chunks))
    
    embeddings = create_embeddings(all_chunks)
    index = build_faiss_index(np.array(embeddings).astype("float32"))
    relevant_chunks = retrieve(index, all_chunks, query)

    ranked_chunks = rerank(relevant_chunks, query)

    results = []
    for chunk in ranked_chunks:
        idx = all_chunks.index(chunk)
        results.append({
            "text": chunk,
            "source": sources[idx]
        })

    context = "\n\n".join([f"{r['text']}\n(Source: {r['source']})" for r in results])

    print(context)

    ans = generate_with_llm(context, query)
    return ans

In [ ]:
query = "How LangChain can be used for creating AI Agents?"
run_pipline(query)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3696.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4610.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4094.32it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
--------------------------

How to Use LangChain to Build Smart AI Agents (With Real Examples) - Why LangChain? LangChain is an open-source orchestration framework for the development of applications using large language models (LLMs). Available in both Python- and Javascript-based libraries, LangChain’s tools and APIs simplify the process of building LLM-driven applications like chatbots and virtual agents. 2. Why I chose to learn it. As someone with a solid foundation in MERN stack development and experience across various tech stacks, I always saw myself as a full-stack developer. But over time, I noticed a shift — AI was no longer just a buzzword; it was becoming the future of tech. That realization kept nudging me. I found myself questioning whether sticking solely to web development was enough. Eventually, I decided to embrace the change, and that’s when I dove into LangChain. It felt like the right step to future-proof my skills and explore the endless possibilities AI has to offer. “Ever wondered how apps

' LangChain is a powerful open-source framework designed specifically for developing intelligent agents using large language models (LLMs). Here are the key steps to create an AI agent with LangChain:\n\n1. **Set Up Your Environment**: Install the necessary libraries using pip:\n    ```bash\n    pip install langchain langchain-openai openai python-dotenv\n    ```\n\n2. **Initialize the LLM (Large Language Model)**: Use a model like OpenAI\'s GPT-4 or another compatible model.\n    ```python\n    from langchain_openai import ChatOpenAI\n    llm = ChatOpenAI(model_name="gpt-4", temperature=0)\n    ```\n\n3. **Define Tools**: Create functions that the agent can use, such as searching the web or performing calculations.\n    ```python\n    @tool\n    def get_weather(city: str) -> str:\n        url = f"https://wttr.in/{city}?format=3"\n        return requests.get(url).text\n\n    tools = [get_weather]\n    ```\n\n4. **Initialize the Agent**: Combine the LLM and tools into an agent using a p

In [78]:
def run_pipline(query):
    urls = search(query)

    all_chunks = []
    sources = []

    for url in urls:
        html = fetch_pages(url)
        text = extract_text(html)
        if text:
            chunks = list(chunk_text(text))
            all_chunks.extend(chunks)
            sources.extend([url]*len(chunks))
    
    embeddings = create_embeddings(all_chunks)
    index = build_faiss_index(np.array(embeddings).astype("float32"))
    relevant_chunks = retrieve(index, all_chunks, query)

    ranked_chunks = rerank(relevant_chunks, query)

    results = []
    for chunk in ranked_chunks:
        idx = all_chunks.index(chunk)
        results.append({
            "text": chunk,
            "source": sources[idx]
        })

    context = "\n\n".join([f"{r['text']}\n(Source: {r['source']})" for r in results])

    print(context)

    ans = generate_with_llm(context, query)
    return ans

In [79]:
query = "How LangChain can be used for creating AI Agents?"
run_pipline(query)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3316.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3746.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3510.75it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
--------------------------

How to Use LangChain to Build Smart AI Agents (With Real Examples) - Why LangChain? LangChain is an open-source orchestration framework for the development of applications using large language models (LLMs). Available in both Python- and Javascript-based libraries, LangChain’s tools and APIs simplify the process of building LLM-driven applications like chatbots and virtual agents. 2. Why I chose to learn it. As someone with a solid foundation in MERN stack development and experience across various tech stacks, I always saw myself as a full-stack developer. But over time, I noticed a shift — AI was no longer just a buzzword; it was becoming the future of tech. That realization kept nudging me. I found myself questioning whether sticking solely to web development was enough. Eventually, I decided to embrace the change, and that’s when I dove into LangChain. It felt like the right step to future-proof my skills and explore the endless possibilities AI has to offer. “Ever wondered how apps

' LangChain is a powerful open-source framework that simplifies the process of building intelligent agents using large language models (LLMs). Here\'s how it can be used to create AI Agents:\n\n1. **Setting Up Your Environment**: Install necessary libraries like `langchain`, `openai`, etc., and set up your environment with API keys.\n   ```bash\n   pip install langchain openai\n   export OPENAI_API_KEY="your-api-key"\n   ```\n\n2. **Creating a Chain**: Define a prompt template and create an LLMChain using the language model (e.g., OpenAI) and the defined prompt.\n   ```python\n   from langchain.llms import OpenAI\n   llm = OpenAI(temperature=0.7)\n   from langchain.prompts import PromptTemplate\n   prompt = PromptTemplate.from_template("What are the top 3 facts about {topic}?")\n   chain = LLMChain(llm=llm, prompt=prompt)\n   ```\n\n3. **Adding Memory**: Use ConversationBufferMemory to store and retrieve conversation history.\n   ```python\n   from langchain.memory import ConversationB